# CAL-02 | Реальные E57: станции и локальная толщина стен

Этот блокнот работает с двумя исходными E57 непосредственно в Google Colab, потому что каждый файл превышает лимит скачивания через подключение ChatGPT. Он **не модифицирует** файлы и **не переписывает** параметры синтетического сканера.

Сначала проверяется структура `data3D` и наличие `rowIndex`, `columnIndex`, `pose`. Затем потоково выбираются точки и оценивается локальный разброс измеренной поверхности около плоскостей стен.

**Важное ограничение:** отклонение точек от плоскости включает неровности стен и ошибки регистрации. Это не чистый шум лазерного дальномера. Один узел `data3D` не равен доказанно одной станции.

In [ ]:
!pip -q install "gdown>=5.2,<6" "numpy>=1.26,<3" "pytest>=8,<10"
import os, subprocess, sys, json, hashlib, shutil
from pathlib import Path
ROOT = Path('/content/E57toARCHICAD')
if not ROOT.exists():
    subprocess.run(['git','clone','-q','--branch','fix/native-hd-v4','https://github.com/BykovVadimCV/E57toARCHICAD.git',str(ROOT)],check=True)
else:
    subprocess.run(['git','-C',str(ROOT),'fetch','-q','origin','fix/native-hd-v4'],check=True)
    subprocess.run(['git','-C',str(ROOT),'checkout','-q','fix/native-hd-v4'],check=True)
    subprocess.run(['git','-C',str(ROOT),'pull','-q','--ff-only'],check=True)
COMMIT=subprocess.check_output(['git','-C',str(ROOT),'rev-parse','HEAD'],text=True).strip()
print('CAL-02 source commit:', COMMIT)
assert (ROOT/'tools/calibrate_e57_surfaces.py').exists(), 'CAL-02 not yet on branch'


## 1. Исходные E57

Если E57 уже лежат у тебя на диске Colab или в подключённом Google Drive, укажи путь в `EXISTING_DIR` и отключи загрузку. Иначе блокнот скачает файлы по исходным общим ссылкам. Объём двух файлов: около **1,24 ГБ**. Они остаются только в твоей сессии Colab.

In [ ]:
import gdown
DATA = Path('/content/e57_cal02_input')
DATA.mkdir(parents=True, exist_ok=True)
EXISTING_DIR = None  # пример: '/content/drive/MyDrive/E57_originals' (после mount)
DOWNLOAD_SHARED_FILES = True
FILES = [
    dict(name='4012_Mikluhi-Maklaya-3-793.e57', id='13FczZrxi3eRVPvKubWo9WOMrgN6Kqdb9',
         size=491343872, sha256='473fe526d139657baa3654ac9cc82681d5e656ee19da5f6d5c57167aa46d9975'),
    dict(name='Barklaya_14_42.e57', id='1JfTGRgpGdqKho50QKkGoEscgup9aErSn',
         size=749791232, sha256='a7f1352a2ae2458f1ca11c33b220f582070ea7b2c3834a32bc53ba6531b4be95'),
]
paths=[]
for f in FILES:
    p = Path(EXISTING_DIR)/f['name'] if EXISTING_DIR else DATA/f['name']
    if not p.exists():
        if not DOWNLOAD_SHARED_FILES:
            raise FileNotFoundError(p)
        print('Загрузка:', f['name'])
        result=gdown.download(id=f['id'],output=str(p),quiet=False)
        if not result or not p.exists():
            raise RuntimeError('Не удалось получить E57: нужен путь из Google Drive в EXISTING_DIR')
    actual=p.stat().st_size
    if actual != f['size']:
        raise ValueError(f'Неверный размер {p.name}: {actual} вместо {f["size"]}')
    h=hashlib.sha256()
    with p.open('rb') as reader:
        while True:
            chunk=reader.read(8*1024*1024)
            if not chunk: break
            h.update(chunk)
    actual_hash=h.hexdigest()
    if actual_hash != f['sha256']:
        raise ValueError(f'SHA256 НЕ совпал: {p.name}')
    print('Подтверждено:', p.name, 'SHA256',actual_hash)
    paths.append(p)


## 2. Быстрая инспекция структуры

Читается только E57-заголовок, а не десятки миллионов точек. Проверь наличие сеточных индексов и поз. Эти метаданные сами по себе **не восстанавливают исходные станции**, если облако было объединено.

In [ ]:
OUT=Path('/content/E57_CAL02_results'); OUT.mkdir(exist_ok=True)
env=dict(os.environ,PYTHONPATH=str(ROOT))
subprocess.run([sys.executable,str(ROOT/'tools/calibrate_e57_surfaces.py'),
                '--e57',*[str(p) for p in paths],'--metadata-only',
                '--out',str(OUT/'metadata')],cwd=str(ROOT),env=env,check=True)
meta=json.loads((OUT/'metadata/cal02_surface_calibration.json').read_text())
for doc in meta['files']:
    print('\n',doc['file_name'],'data3D nodes:',doc['data3d_node_count'])
    for s in doc['scans']:
        print('fields:',s['fields'])
        print('row/column:',s['row_and_column_indices_available'],
              'pose:',s['pose_rotation_available'] and s['pose_translation_available'])


## 3. Потоковая калибровка наблюдаемых поверхностей

Читаются исходные XYZ по частям и создаётся ограниченная случайная выборка до **250 000 точек на узел data3D**. По ней находятся кандидаты вертикальных плоскостей и оценивается их локальный разброс. Полный E57 не загружается в память. Если поверхность не удалось надёжно обнаружить, результат останется пустым.

Это **не** оценка `range_sigma_mm`. Никакие изменения в `customer_like.yaml` не выполняются.

In [ ]:
subprocess.run([sys.executable,str(ROOT/'tools/calibrate_e57_surfaces.py'),
                '--e57',*[str(p) for p in paths],
                '--out',str(OUT/'surfaces'), '--max-sample','250000',
                '--chunk','250000','--max-lines','5', '--max-patches','24',
                '--seed','42'],cwd=str(ROOT),env=env,check=True)
print((OUT/'surfaces/CAL02_REPORT.md').read_text(encoding='utf8'))


## 4. Упаковка проверяемого отчёта

В архиве будут только JSON/Markdown и номер исходного Git-коммита. Исходные E57 и выборки точек не включаются. Архив можно загрузить обратно в чат для анализа реальных измерений.

In [ ]:
(OUT/'SOURCE_COMMIT.txt').write_text(COMMIT+'\n')
(OUT/'original_sha256.json').write_text(json.dumps([
    {'file':f['name'],'sha256':f['sha256'],'size_bytes':f['size']} for f in FILES],indent=2))
archive=shutil.make_archive('/content/E57_CAL02_real_surface_results','zip',root_dir=str(OUT))
print('Готовый отчёт:',archive,Path(archive).stat().st_size,'bytes')
from google.colab import files
files.download(archive)
